In [2]:
import os
import shutil
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================================
# 1. CONFIGURARE CAI (Diabetic_Balanced_Aug)
# ==========================================
BASE_DIR = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\originals\Diabetic_Balanced_Aug")
CSV_PATH = BASE_DIR / "trainLabels.csv"
IMG_FOLDER = BASE_DIR / "resized_train" / "resized_train"

# Destinatia datasetului curat (impartit)
OUTPUT_DIR = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\balanced_aug\balanced_aug_split")

CLASSES = [0, 1, 2, 3, 4]
WORKERS = max((os.cpu_count() or 2) - 1, 1)

RATIO_TRAIN = 0.70
RATIO_VAL = 0.20
RATIO_TEST = 0.10

In [3]:
# ==========================================
# 2. FUNCTIE PENTRU COPIEREA RAPIDA
# ==========================================
def copy_file(task):
    src, dst = task
    try:
        shutil.copy2(src, dst)  # Pastreaza metadatele si este foarte rapid
        return True, ""
    except Exception as e:
        return False, f"Eroare copiere {src.name}: {e}"

In [4]:
# ==========================================
# 3. EXECUTIA PRINCIPALA
# ==========================================
def main():
    if OUTPUT_DIR.exists():
        print("Curatam folderul de output existent...")
        shutil.rmtree(OUTPUT_DIR)

    # Cream arhitectura de foldere (train/0, val/0, test/0 etc.)
    for split in ['train', 'val', 'test']:
        for cls in CLASSES:
            (OUTPUT_DIR / split / str(cls)).mkdir(parents=True, exist_ok=True)

    # 1. Citim CSV-ul
    if not CSV_PATH.exists():
        print(f"Eroare: Nu s-a gasit CSV-ul la {CSV_PATH}")
        return

    df = pd.read_csv(CSV_PATH)
    
    # 2. Verificam care imagini exista cu adevarat pe disc
    valid_data = []
    print("\n--- 1. Verificam imaginile pe disc ---")
    
    if not IMG_FOLDER.exists():
        print(f"Eroare: Folderul de imagini {IMG_FOLDER} nu exista.")
        return
        
    # Incarcam in memorie ce avem in folder pentru cautare instantanee (O(1))
    fisiere_existente = {f.name: f for f in IMG_FOLDER.iterdir() if f.is_file()}
    
    # Coloanele din trainLabels.csv sunt 'image' si 'level'
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Scanare CSV"):
        img_id = str(row['image'])
        label = int(row['level'])
        
        found_path = None
        # Incercam cu si fara extensie in numele din CSV, uzual se adauga .jpeg
        for ext in ['.jpeg', '.jpg', '.png', '.tif']:
            nume_fisier = f"{img_id}{ext}"
            if nume_fisier in fisiere_existente:
                found_path = fisiere_existente[nume_fisier]
                break
                
        if found_path:
            valid_data.append({'image_path': found_path, 'label': label, 'image_id': img_id})

    df_valid = pd.DataFrame(valid_data)
    print(f"\n✅ Total imagini gasite si valide: {len(df_valid)}")

    if len(df_valid) == 0:
        print("Nu am gasit nicio imagine. Verifica te rog calea folderului 'resized_train/resized_train'.")
        return

    # 3. Impartirea Stratificata 70% / 20% / 10%
    print("\n--- 2. Calculam impartirea stratificata ---")
    
    # Pasul A: Extragem Test-ul (10%) din total
    df_train_val, df_test = train_test_split(
        df_valid, 
        test_size=RATIO_TEST, 
        stratify=df_valid['label'], 
        random_state=42
    )
    
    # Pasul B: Impartim restul (90%) in Train (70% din total) si Val (20% din total)
    val_fraction = RATIO_VAL / (RATIO_TRAIN + RATIO_VAL)
    df_train, df_val = train_test_split(
        df_train_val, 
        test_size=val_fraction, 
        stratify=df_train_val['label'], 
        random_state=42
    )

    print(f"Distributie finala:\n TRAIN: {len(df_train)} imagini\n VAL:   {len(df_val)} imagini\n TEST:  {len(df_test)} imagini")

    # 4. Generam lista de task-uri pentru copiere
    tasks = []
    
    def add_tasks(dataframe, split_name):
        for _, row in dataframe.iterrows():
            src = row['image_path']
            lbl = row['label']
            img_id = row['image_id']
            dst = OUTPUT_DIR / split_name / str(lbl) / f"{img_id}{src.suffix}"
            tasks.append((src, dst))

    add_tasks(df_train, 'train')
    add_tasks(df_val, 'val')
    add_tasks(df_test, 'test')

    # 5. Rulam multithreading-ul pentru viteza
    print(f"\n--- 3. Incepem copierea fisierelor ({len(tasks)} operatiuni) ---")
    processed, errors = 0, 0

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = [executor.submit(copy_file, task) for task in tasks]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Copiere Foldere"):
            ok, msg = future.result()
            if ok: processed += 1
            else: 
                errors += 1
                print(f"\n[Eroare] {msg}")

    print("\n" + "="*50)
    print("FINALIZAT! Datasetul Balanced_Aug a fost impartit cu succes.")
    print(f"Locatia noului dataset: {OUTPUT_DIR}")

In [5]:
if __name__ == '__main__':
    main()


--- 1. Verificam imaginile pe disc ---


Scanare CSV: 100%|██████████| 35126/35126 [00:00<00:00, 59531.39it/s]



✅ Total imagini gasite si valide: 35126

--- 2. Calculam impartirea stratificata ---
Distributie finala:
 TRAIN: 24587 imagini
 VAL:   7026 imagini
 TEST:  3513 imagini

--- 3. Incepem copierea fisierelor (35126 operatiuni) ---


Copiere Foldere: 100%|██████████| 35126/35126 [00:16<00:00, 2146.82it/s]


FINALIZAT! Datasetul Balanced_Aug a fost impartit cu succes.
Locatia noului dataset: B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\balanced_aug\balanced_aug_split
